In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries Loaded!")

In [ ]:
housing = pd.read_csv('Housing.csv')

print("✅ Dataset Loaded!")
print("Shape:", housing.shape)
housing.head()

In [ ]:
print(housing.info())
print("\n=== STATISTICS ===")
print(housing.describe())

print("\n=== MISSING VALUES (%) ===")
print(housing.isnull().sum() * 100 / housing.shape[0])
print("\n✅ No missing values - dataset is clean!")

In [ ]:
# Outlier treatment for price
plt.figure(figsize=(6,4))
sns.boxplot(housing['price'])
plt.title('Price - Before Outlier Removal')
plt.show()

Q1 = housing.price.quantile(0.25)
Q3 = housing.price.quantile(0.75)
IQR = Q3 - Q1
housing = housing[(housing.price >= Q1 - 1.5*IQR) & (housing.price <= Q3 + 1.5*IQR)]

# Outlier treatment for area
plt.figure(figsize=(6,4))
sns.boxplot(housing['area'])
plt.title('Area - Before Outlier Removal')
plt.show()

Q1 = housing.area.quantile(0.25)
Q3 = housing.area.quantile(0.75)
IQR = Q3 - Q1
housing = housing[(housing.area >= Q1 - 1.5*IQR) & (housing.area <= Q3 + 1.5*IQR)]

print("✅ Outliers Removed!")
print("Shape after outlier removal:", housing.shape)

In [ ]:
sns.pairplot(housing)
plt.savefig('chart1_pairplot.png', dpi=150)
plt.show()
print("✅ Chart 1 Saved!")

In [ ]:
plt.figure(figsize=(20, 12))
plt.subplot(2,3,1)
sns.boxplot(x = 'mainroad', y = 'price', data = housing)
plt.subplot(2,3,2)
sns.boxplot(x = 'guestroom', y = 'price', data = housing)
plt.subplot(2,3,3)
sns.boxplot(x = 'basement', y = 'price', data = housing)
plt.subplot(2,3,4)
sns.boxplot(x = 'hotwaterheating', y = 'price', data = housing)
plt.subplot(2,3,5)
sns.boxplot(x = 'airconditioning', y = 'price', data = housing)
plt.subplot(2,3,6)
sns.boxplot(x = 'furnishingstatus', y = 'price', data = housing)
plt.savefig('chart2_categorical_boxplots.png', dpi=150)
plt.show()
print("✅ Chart 2 Saved!")

In [ ]:
plt.figure(figsize = (10, 5))
sns.boxplot(x = 'furnishingstatus', y = 'price', hue = 'airconditioning', data = housing)
plt.title('Furnishing Status vs Price (by Air Conditioning)', fontsize=14, fontweight='bold')
plt.savefig('chart3_furnishing_ac.png', dpi=150)
plt.show()
print("✅ Chart 3 Saved!")

In [ ]:
# List of variables to map
varlist = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

# Defining the map function
def binary_map(x):
    return x.map({'yes': 1, "no": 0})

# Applying the function to the housing list
housing[varlist] = housing[varlist].apply(binary_map)

print("✅ Binary Mapping Done!")
housing.head()

In [ ]:
# Get dummy variables and drop first column (avoid multicollinearity)
status = pd.get_dummies(housing['furnishingstatus'], drop_first=True)
status = status.astype(int)

# Add the results to the original housing dataframe
housing = pd.concat([housing, status], axis=1)

# Drop the original 'furnishingstatus' column
housing = housing.drop(['furnishingstatus'], axis=1)

print("✅ Dummy Variables Created!")
housing.head()

In [ ]:
from sklearn.model_selection import train_test_split

np.random.seed(0)
df_train, df_test = train_test_split(housing, train_size=0.7, test_size=0.3, random_state=100)

print("✅ Train-Test Split Done!")
print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

num_vars = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'price']

df_train[num_vars] = scaler.fit_transform(df_train[num_vars])

print("✅ Scaling Done!")
df_train.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

num_vars = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'price']

df_train[num_vars] = scaler.fit_transform(df_train[num_vars])

print("✅ Scaling Done!")
df_train.head()

In [ ]:
plt.figure(figsize=(16, 10))
sns.heatmap(df_train.corr(), annot=True, cmap="YlGnBu")
plt.title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.savefig('chart4_correlation_heatmap.png', dpi=150)
plt.show()
print("✅ Chart 4 Saved!")

In [ ]:
y_train = df_train.pop('price')
X_train = df_train

print("✅ Features and Target Split!")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

rfe = RFE(estimator=lm, n_features_to_select=6)
rfe = rfe.fit(X_train, y_train)

print("✅ RFE Feature Selection Done!")
print("\nSelected Features:")
for feature, selected, rank in zip(X_train.columns, rfe.support_, rfe.ranking_):
    print(f"{feature}: Selected={selected}, Rank={rank}")

In [ ]:
import statsmodels.api as sm

col = X_train.columns[rfe.support_]

X_train_rfe = X_train[col]
X_train_lm = sm.add_constant(X_train_rfe)

lr_model = sm.OLS(y_train, X_train_lm).fit()

print(lr_model.summary())

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif['Features'] = X_train_rfe.columns
vif['VIF'] = [variance_inflation_factor(X_train_rfe.values, i) for i in range(X_train_rfe.shape[1])]
vif['VIF'] = round(vif['VIF'], 2)
vif = vif.sort_values(by="VIF", ascending=False)

print("✅ VIF Calculated!")
print(vif)

In [ ]:
y_train_price = lr_model.predict(X_train_lm)

fig = plt.figure(figsize=(8,5))
sns.histplot((y_train - y_train_price), bins=20, kde=True)
plt.title('Error Terms Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Errors (y_train - y_train_pred)', fontsize=12)
plt.tight_layout()
plt.savefig('chart5_residuals.png', dpi=150)
plt.show()
print("✅ Chart 5 Saved!")

In [ ]:
# Apply scaling on test set
df_test[num_vars] = scaler.transform(df_test[num_vars])

y_test = df_test.pop('price')
X_test = df_test

# Use the selected features
X_test_rfe = X_test[col]
X_test_lm = sm.add_constant(X_test_rfe)

y_pred = lr_model.predict(X_test_lm)

print("✅ Predictions Made!")

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("=" * 50)
print("📊 MODEL EVALUATION")
print("=" * 50)
print(f"R-squared (Train): {round(lr_model.rsquared, 4)}")
print(f"R-squared (Test): {round(r2, 4)}")
print(f"Mean Squared Error: {round(mse, 6)}")

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.6, color='#4ECDC4')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         color='red', linestyle='--', linewidth=2, label='Perfect Prediction')
plt.title('Actual vs Predicted House Prices', fontsize=16, fontweight='bold')
plt.xlabel('Actual Prices (Scaled)', fontsize=12)
plt.ylabel('Predicted Prices (Scaled)', fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig('chart6_actual_vs_predicted.png', dpi=150)
plt.show()
print("✅ Chart 6 Saved!")

In [ ]:
print("=" * 50)
print("📊 KEY INSIGHTS & RECOMMENDATIONS")
print("=" * 50)

print(f"\n1. 🎯 Model R-squared (Test): {round(r2*100, 2)}%")
print(f"2. 📈 Selected Features: {list(col)}")
print(f"3. 💰 Price strongly depends on: area, bathrooms, stories")

print("\n📌 RECOMMENDATIONS:")
print("   → Area and number of bathrooms are top price drivers")
print("   → Air conditioning and preferred area increase value")
print("   → Model explains a good portion of price variance")

In [ ]:
from google.colab import files

files.download('chart1_pairplot.png')
files.download('chart2_categorical_boxplots.png')
files.download('chart3_furnishing_ac.png')
files.download('chart4_correlation_heatmap.png')
files.download('chart5_residuals.png')
files.download('chart6_actual_vs_predicted.png')

print("✅ All Charts Downloaded!")